# First steps with the new code
Lets try the new Trefftzmesh

In [2]:
from trefftz.mesh import TrefftzMesh
from trefftz.mesh.readers import GmshArrays

In [3]:
from enum import IntEnum
import gmsh

class WaveguideRegions(IntEnum):
    OMEGA = 0
    GAMMA = 1
    SIGMA_L = 2
    SIGMA_R = 3

def CleanWaveguide(H: float = 1., R: float = 5., lc: float = 0.3, verbose: bool = False) -> TrefftzMesh[WaveguideRegions]:

    '''Creates a domain corresponging to a waveguide without scatterers.
    It assumes the default tags for the subregions, i.e.:
    - Omega = 0
    - Gamma = 1
    - Sigma = 2
    '''

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", int(verbose))
    gmsh.model.add("Waveguide")
    p0 = gmsh.model.geo.addPoint(-R, 0., 0., lc)
    p1 = gmsh.model.geo.addPoint( R, 0., 0., lc)
    p2 = gmsh.model.geo.addPoint( R,  H, 0., lc)
    p3 = gmsh.model.geo.addPoint(-R,  H, 0., lc)

    bottom = gmsh.model.geo.addLine(p0, p1)
    right  = gmsh.model.geo.addLine(p1, p2)
    top    = gmsh.model.geo.addLine(p2, p3)
    left   = gmsh.model.geo.addLine(p3, p0)

    boundary = gmsh.model.geo.addCurveLoop([bottom, right, top, left])
    domain = gmsh.model.geo.addPlaneSurface([boundary])
    gmsh.model.geo.synchronize()

    gmsh.model.addPhysicalGroup(2, [domain], WaveguideRegions.OMEGA, "Omega")
    gmsh.model.addPhysicalGroup(1, [bottom, top], WaveguideRegions.GAMMA, "Gamma")
    gmsh.model.addPhysicalGroup(1, [left], WaveguideRegions.SIGMA_L, "Sigma_L")
    gmsh.model.addPhysicalGroup(1, [right], WaveguideRegions.SIGMA_R, "Sigma_R")
    
    gmsh.model.geo.synchronize()
    gmsh.model.mesh.generate(2)
    # gmsh.write('CleanWaveguide.msh')
    # self._domain = TrefftzMesh.from_gmsh('CleanWaveguide.msh')
    points, edges, triangles, edges2triangles, locator, cell_sets = GmshArrays(gmsh.model)
    gmsh.finalize()

    mesh = TrefftzMesh(points, edges, triangles, WaveguideRegions, edges2triangles, locator, cell_sets)

    return mesh


In [5]:
mesh = CleanWaveguide(verbose=True)

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000198423s, CPU 0.000147s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.00372449s, CPU 0s)
Info    : 181 nodes 364 elements


In [9]:
from problems.base2 import Problem

In [10]:
from trefftz.dg.basis import LinearlySpacedBasis

k = 8.0

basis = LinearlySpacedBasis(N_elements=mesh.n_triangles, k=k, N_theta=10)

In [11]:
from problems.base2 import SoundHardBC, RadiatingBC

In [12]:
boundary_conditions = {
    WaveguideRegions.GAMMA: SoundHardBC(d_1=0.5),
    WaveguideRegions.SIGMA_L: RadiatingBC(),
    WaveguideRegions.SIGMA_R: RadiatingBC(),
}

In [13]:
P = Problem(mesh=mesh, wavenumber=k, basis=basis, regions=WaveguideRegions, boundary_conditions=boundary_conditions)